In [1]:
import itertools
from multiprocessing.pool import Pool

import pandas as pd
from sklearn import metrics
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april.anomalydetection import BINet
from april.anomalydetection.utils import label_collapse
from april.database import Evaluation
from april.database import Model
from april.database import get_engine
from april.enums import Base
from april.enums import Heuristic
from april.enums import Strategy
from april.evaluator import Evaluator
from april.fs import get_model_files
from april.fs import PLOT_DIR

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set
Creating Evaluation table


In [2]:
heuristics = [h for h in Heuristic.keys() if h not in [Heuristic.DEFAULT, Heuristic.MANUAL, Heuristic.RATIO,
                                                       Heuristic.MEDIAN, Heuristic.MEAN]]
params = [(Base.SCORES, Heuristic.DEFAULT, Strategy.SINGLE), *itertools.product([Base.SCORES], heuristics, Strategy.keys())]

In [3]:
def _evaluate(params):
    # print(f"Evaluating {params}...")
    e, base, heuristic, strategy = params

    session = Session(get_engine())
    model = session.query(Model).filter_by(file_name=e.model_file.name).first()
    session.close()
    # print(f"Model {model} loaded from Session.")

    if Model is None:
        print(f"Models from session: {model}")
    # Generate evaluation frames
    y_pred = e.binarizer.binarize(base=base, heuristic=heuristic, strategy=strategy, go_backwards=False)
    y_true = e.binarizer.get_targets()

    evaluations = []
    for axis in [0, 1, 2]:
        # print(f"Evaluating axis {axis}...")
        for i, attribute_name in enumerate(e.dataset.attribute_keys):
            # print(f"Evaluating attribute {attribute_name}...")
            def get_evaluation(label, precision, recall, f1):
                return Evaluation(model_id=model.id, file_name=model.file_name,
                                  label=label, perspective=perspective, attribute_name=attribute_name,
                                  axis=axis, base=base, heuristic=heuristic, strategy=strategy,
                                  precision=precision, recall=recall, f1=f1)

            perspective = 'Control Flow' if i == 0 else 'Data'
            if i > 0 and not e.ad_.supports_attributes:
                # print(f"Skipping attribute {attribute_name} for model {model} as it does not support attributes.")
                evaluations.append(get_evaluation('Normal', 0.0, 0.0, 0.0))
                evaluations.append(get_evaluation('Anomaly', 0.0, 0.0, 0.0))
            else:
                # print(f"Evaluating attribute {attribute_name} for model {model}...")
                yp = label_collapse(y_pred[:, :, i:i + 1], axis=axis).compressed()
                yt = label_collapse(y_true[:, :, i:i + 1], axis=axis).compressed()
                p, r, f, _ = metrics.precision_recall_fscore_support(yt, yp, labels=[0, 1])
                evaluations.append(get_evaluation('Normal', p[0], r[0], f[0]))
                evaluations.append(get_evaluation('Anomaly', p[1], r[1], f[1]))

    return evaluations

def evaluate(model_name):
    # print(f"Evaluating {model_name}...")
    e = Evaluator(model_name)
    # print(f"{e} loaded.")

    _params = []
    for base, heuristic, strategy in params:
        if e.dataset.num_attributes == 1 and strategy in [Strategy.ATTRIBUTE, Strategy.POSITION_ATTRIBUTE]:
            continue
        if isinstance(e.ad_, BINet) and e.ad_.version == 0:
            continue
        if heuristic is not None and heuristic not in e.ad_.supported_heuristics:
            continue
        if strategy is not None and strategy not in e.ad_.supported_strategies:
            continue
        if base is not None and base not in e.ad_.supported_bases:
            continue
        _params.append([e, base, heuristic, strategy])

    return [_e for p in _params for _e in _evaluate(p)]

In [4]:
models = sorted([m.name for m in get_model_files() if m.p == 0.3])# and 'real' in m.name])
print(models)
evaluations = []
# with Pool() as p:
#     for e in tqdm(p.imap(evaluate, models), total=len(models), desc='Evaluate'):
#         evaluations.append(e)
for model in tqdm(models, desc='Evaluate'):
    e = evaluate(model)
    evaluations.append(e)
# Write to database
session = Session(get_engine())
for e in evaluations:
    session.bulk_save_objects(e)
    session.commit()
session.close()

['paper-0.3-1_dae_20250424-134746.449773', 'paper-0.3-1_daeltnfrozen_20250424-134751.674323']


Evaluate:   0%|          | 0/2 [00:00<?, ?it/s]

Loading model paper-0.3-1_dae_20250424-134746.449773 for event log paper-0.3-1
Filtering dataset to 669 LTN rows.
Indices: [1, 4, 6, 15, 17, 32, 61, 62, 63, 75, 77, 95, 96, 97, 100, 106, 108, 116, 124, 132, 136, 138, 147, 148, 149, 158, 166, 176, 180, 184, 190, 215, 226, 237, 239, 245, 247, 257, 258, 262, 264, 269, 270, 281, 283, 286, 288, 294, 324, 329, 330, 350, 357, 358, 376, 391, 417, 421, 423, 433, 435, 436, 439, 441, 447, 454, 475, 478, 483, 484, 487, 488, 492, 502, 517, 520, 524, 534, 563, 565, 567, 587, 590, 597, 619, 625, 628, 636, 662, 663, 677, 684, 691, 702, 723, 725, 747, 750, 770, 783, 785, 787, 789, 815, 822, 823, 833, 834, 835, 837, 844, 852, 854, 855, 858, 865, 868, 870, 874, 894, 898, 900, 906, 917, 920, 921, 926, 927, 950, 966, 972, 974, 995, 1013, 1014, 1026, 1027, 1028, 1029, 1030, 1031, 1033, 1040, 1047, 1053, 1059, 1061, 1064, 1076, 1077, 1087, 1090, 1094, 1104, 1106, 1114, 1116, 1122, 1126, 1127, 1132, 1135, 1144, 1145, 1157, 1158, 1161, 1175, 1211, 1215, 1231, 

## Pickle the results

In [5]:
out_dir = PLOT_DIR / 'isj-2019'
eval_file = out_dir / 'eval.pkl'

session = Session(get_engine())
evaluations = session.query(Evaluation).all()
print(vars(evaluations[0]))
rows = []

for ev in tqdm(evaluations):
    # print(f"Evaluation: {ev}")
    m = ev.model
    # print(f"Model: {m}")
    el = ev.model.training_event_log
    # print(f"Event log: {el}")
    rows.append([m.file_name, m.creation_date, m.hyperparameters, m.training_duration, m.training_host, m.algorithm, 
                 el.name, el.base_name, el.percent_anomalies, el.number,
                 ev.axis, ev.base, ev.heuristic, ev.strategy, ev.label, ev.attribute_name, ev.perspective, ev.precision, ev.recall, ev.f1])
session.close()
columns = ['file_name', 'date', 'hyperparameters', 'training_duration', 'training_host', 'ad',
           'dataset_name', 'process_model', 'noise', 'dataset_id',
           'axis', 'base', 'heuristic', 'strategy', 'label', 'attribute_name', 'perspective', 'precision', 'recall', 'f1']
evaluation = pd.DataFrame(rows, columns=columns)

evaluation.to_pickle(eval_file)

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState object at 0x000001D28BDB3130>, 'recall': 0.8577154308617234, 'attribute_name': 'name', 'perspective': 'Control Flow', 'strategy': 'single', 'base': 'scores', 'file_name': 'paper-0.3-1_dae_20250424-134746.449773.keras', 'id': 1, 'f1': 0.9234088457389428, 'precision': 1.0, 'label': 'Normal', 'heuristic': 'best', 'axis': 0, 'model_id': 1}


  0%|          | 0/576 [00:00<?, ?it/s]

In [6]:
evaluation.to_pickle(eval_file)

In [7]:
display(evaluation)

,file_name,date,hyperparameters,training_duration,training_host,ad,dataset_name,process_model,noise,dataset_id,axis,base,heuristic,strategy,label,attribute_name,perspective,precision,recall,f1
0,paper-0.3-1_dae_20250424-134746.449773.keras,2025-04-24 13:47:51.167952,"{'epochs': 8, 'batch_size': 100}",4.718179,Dev-RTX,DAE,paper-0.3-1,paper,0.3,1,0,scores,best,single,Normal,name,Control Flow,1.000000,0.857715,0.923409
1,paper-0.3-1_dae_20250424-134746.449773.keras,2025-04-24 13:47:51.167952,"{'epochs': 8, 'batch_size': 100}",4.718179,Dev-RTX,DAE,paper-0.3-1,paper,0.3,1,0,scores,best,single,Anomaly,name,Control Flow,0.705394,1.000000,0.827251
2,paper-0.3-1_dae_20250424-134746.449773.keras,2025-04-24 13:47:51.167952,"{'epochs': 8, 'batch_size': 100}",4.718179,Dev-RTX,DAE,paper-0.3-1,paper,0.3,1,0,scores,best,single,Normal,user,Data,0.895366,1.000000,0.944795
3,paper-0.3-1_dae_20250424-134746.449773.keras,2025-04-24 13:47:51.167952,"{'epochs': 8, 'batch_size': 100}",4.718179,Dev-RTX,DAE,paper-0.3-1,paper,0.3,1,0,scores,best,single,Anomaly,user,Data,0.000000,0.000000,0.000000
4,paper-0.3-1_dae_20250424-134746.449773.keras,2025-04-24 13:47:51.167952,"{'epochs': 8, 'batch_size': 100}",4.718179,Dev-RTX,DAE,paper-0.3-1,paper,0.3,1,1,scores,best,single,Normal,name,Control Flow,0.994685,0.914480,0.952898
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,paper-0.3-1_daeltnfrozen_20250424-134751.67432...,2025-04-24 13:50:43.082956,"{'epochs': 8, 'batch_size': 100, 'epochs_ltn': 4}",171.408633,Dev-RTX,DAELTNFROZEN,paper-0.3-1,paper,0.3,1,1,scores,stable_right,position_attribute,Anomaly,user,Data,0.102493,0.272059,0.148893
572,paper-0.3-1_daeltnfrozen_20250424-134751.67432...,2025-04-24 13:50:43.082956,"{'epochs': 8, 'batch_size': 100, 'epochs_ltn': 4}",171.408633,Dev-RTX,DAELTNFROZEN,paper-0.3-1,paper,0.3,1,2,scores,stable_right,position_attribute,Normal,name,Control Flow,0.969151,0.981739,0.975404
573,paper-0.3-1_daeltnfrozen_20250424-134751.67432...,2025-04-24 13:50:43.082956,"{'epochs': 8, 'batch_size': 100, 'epochs_ltn': 4}",171.408633,Dev-RTX,DAELTNFROZEN,paper-0.3-1,paper,0.3,1,2,scores,stable_right,position_attribute,Anomaly,name,Control Flow,0.429719,0.305714,0.357262
574,paper-0.3-1_daeltnfrozen_20250424-134751.67432...,2025-04-24 13:50:43.082956,"{'epochs': 8, 'batch_size': 100, 'epochs_ltn': 4}",171.408633,Dev-RTX,DAELTNFROZEN,paper-0.3-1,paper,0.3,1,2,scores,stable_right,position_attribute,Normal,user,Data,0.987250,0.959449,0.973151


In [8]:
# print current time and date wih imports
import datetime
now = datetime.datetime.now()
print("Current date and time: ", now.strftime("%Y-%m-%d %H:%M:%S"))
print("Done")

Current date and time:  2025-04-24 13:51:21
Done
